# Geordie Miner — explore a run

Interactive view of a single `output/<name>/` directory.

Edit the `RUN` constant below if you want a different run, then **Run All Cells**.

In [ ]:
import os
import glob
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

# Auto-pick the most recently modified run in ../output/ if RUN is left None.
RUN = None  # e.g. "fulltext_noreference_nomethodology"

OUTPUT_BASE = Path("..") / "output"
if RUN is None:
    runs = sorted(OUTPUT_BASE.glob("*"), key=lambda p: p.stat().st_mtime, reverse=True)
    runs = [r for r in runs if r.is_dir()]
    assert runs, f"No subdirectories under {OUTPUT_BASE}/ — run the pipeline first."
    RUN = runs[0].name

RUN_DIR = OUTPUT_BASE / RUN
print(f"Inspecting: {RUN_DIR}")

## Corpus statistics

In [ ]:
stats_path = RUN_DIR / "corpus_stats.txt"
if stats_path.exists():
    print(stats_path.read_text(encoding="utf-8"))
else:
    print("(no corpus_stats.txt)")

## Top terms

In [ ]:
terms_path = RUN_DIR / "terms_lemmatised.csv"
if terms_path.exists():
    display(pd.read_csv(terms_path).head(30))
else:
    print("(no terms_lemmatised.csv)")

## Top phrases

In [ ]:
for name in ("bigrams.csv", "trigrams.csv"):
    p = RUN_DIR / name
    if p.exists():
        display(Markdown(f"### {name}"))
        display(pd.read_csv(p).head(15))

## Word clouds

In [ ]:
for name in ("wordcloud_lemmatised.jpg", "wordcloud_raw.jpg"):
    p = RUN_DIR / name
    if p.exists():
        display(Markdown(f"### {name}"))
        display(Image(filename=str(p)))

## Topic models

In [ ]:
for p in sorted(RUN_DIR.glob("topics_*.txt")):
    display(Markdown(f"### {p.name}"))
    print(p.read_text(encoding="utf-8"))

## Unified topic assignments

In [ ]:
p = RUN_DIR / "topic_assignments.csv"
if p.exists():
    display(pd.read_csv(p).head(20))

## Top documents per topic

In [ ]:
p = RUN_DIR / "topic_top_docs.csv"
if p.exists():
    df = pd.read_csv(p)
    # Show one example: top docs for first model at lowest K
    if not df.empty:
        first = df.iloc[0]
        subset = df[(df["model"] == first["model"]) & (df["K"] == first["K"])]
        display(Markdown(f"### {first['model']} (K={int(first['K'])})"))
        display(subset)

## Coherence scores

In [ ]:
p = RUN_DIR / "coherence_scores.csv"
if p.exists():
    display(pd.read_csv(p))
else:
    print("(no coherence_scores.csv — only produced after the topics stage)")

## Dendrogram + network

In [ ]:
dendro = RUN_DIR / "dendrogram.png"
if dendro.exists():
    display(Image(filename=str(dendro)))

gexf = RUN_DIR / "network.gexf"
if gexf.exists():
    import networkx as nx
    g = nx.read_gexf(str(gexf))
    print(f"Co-occurrence network: {g.number_of_nodes()} nodes, {g.number_of_edges()} edges")
    print("Open in Gephi for visual exploration: ", gexf.resolve())